In [1]:
try:
    import cirq
except ImportError:
    print("installing cirq...")
    !pip install cirq --quiet
    import cirq
    print("installed cirq.")

print("Libraries Successfully Imported")

Libraries Successfully Imported


In [8]:
class FaultTolerance:
    """Tests whether one fault stays within the code's correction limit."""

    def __init__(self, code_distance=3):
        #distance 3 can correct one arbitrary data-qubit error
        self.code_distance = code_distance
        self.max_correctable_errors = (code_distance - 1) // 2

    def count_errors(self, data_errors):
        """Counts how many data qubits have errors."""

        return len(data_errors)

    def is_correctable(self, data_errors):
        """Checks if the error count stays within the code's limit."""

        return self.count_errors(data_errors) <= self.max_correctable_errors

    def analyze_errors(self, data_errors):
        """Explains whether one final error pattern is correctable."""

        number_of_errors = self.count_errors(data_errors)

        print("data errors:", data_errors)
        print("number of data errors:", number_of_errors)
        print("maximum correctable errors:", self.max_correctable_errors)

        if self.is_correctable(data_errors):
            print("result: the errors stay within the correction limit")
        else:
            print("result: the fault spread past the correction limit")

    def test_single_fault(self, propagation, circuit, fault_step, error):
        """Tests one ancilla fault at one circuit step."""

        data_errors = propagation.run_with_ancilla_error(
            circuit=circuit,
            fault_after_step=fault_step,
            error=error
        )

        return {
            "fault_step": fault_step,
            "fault_type": error,
            "data_errors": data_errors,
            "number_of_errors": self.count_errors(data_errors),
            "correctable": self.is_correctable(data_errors)
        }

    def test_all_single_faults(self, propagation, circuit):
        """Tests X, Y, and Z ancilla faults after every operation."""

        results = []
        number_of_steps = sum(len(moment.operations) for moment in circuit)

        for error in ["X", "Y", "Z"]:
            for fault_step in range(1, number_of_steps + 1):
                result = self.test_single_fault(
                    propagation=propagation,
                    circuit=circuit,
                    fault_step=fault_step,
                    error=error
                )

                results.append(result)

        return results

    def circuit_is_fault_tolerant(self, results):
        """Checks if every tested single fault stays correctable."""

        return all(result["correctable"] for result in results)

    def show_results(self, results):
        """Prints one line for every fault test."""

        print("single-fault test results")
        print("-------------------------")

        for result in results:
            print(
                f"step {result['fault_step']} | "
                f"fault {result['fault_type']} | "
                f"data errors {result['number_of_errors']} | "
                f"correctable {result['correctable']}"
            )

    def analyze_circuit(self, propagation, circuit):
        """Tests every single ancilla fault in the circuit."""

        results = self.test_all_single_faults(propagation, circuit)
        self.show_results(results)

        print()
        print("code distance:", self.code_distance)
        print("maximum correctable errors:", self.max_correctable_errors)

        if self.circuit_is_fault_tolerant(results):
            print("final result: circuit passes the single-fault test")
        else:
            print("final result: circuit fails the single-fault test")

        return results

In [9]:
fault_tolerance = FaultTolerance(code_distance=3)

print("test 1: no data errors")
no_errors = {}
fault_tolerance.analyze_errors(no_errors)

print()

print("test 2: one data error")
one_error = {"q0": "X"}
fault_tolerance.analyze_errors(one_error)

print()

print("test 3: three data errors from one ancilla fault")
three_errors = {
    "q0": "X",
    "q1": "X",
    "q2": "X"
}
fault_tolerance.analyze_errors(three_errors)

test 1: no data errors
data errors: {}
number of data errors: 0
maximum correctable errors: 1
result: the errors stay within the correction limit

test 2: one data error
data errors: {'q0': 'X'}
number of data errors: 1
maximum correctable errors: 1
result: the errors stay within the correction limit

test 3: three data errors from one ancilla fault
data errors: {'q0': 'X', 'q1': 'X', 'q2': 'X'}
number of data errors: 3
maximum correctable errors: 1
result: the fault spread past the correction limit
